# TDT-FL — 03. Modèles du régime statique (R1) et expérience E0 — version 1.0

Ce notebook entraîne, pour chaque graine, tous les modèles de base du régime R1 et mesure **E0 : y a-t-il de la marge entre le modèle local et l'apprentissage fédéré, surtout dans les petits hôpitaux ?**

| Étape | Contenu | Sorties |
|---|---|---|
| 0 | Configuration | — |
| 1 | Modules `tdtfl.models` et `tdtfl.metrics` | `src/tdtfl/` |
| 2 | Données : protocole figé, tâche, chargement par graine | — |
| 3 | Modèles locaux (régression logistique L2, un par hôpital) et prédictions hors plis | `outputs/models/<tâche>/local_r1_s*.npz` |
| 4 | Modèle centralisé (toutes les données d'entraînement réunies) | `central_r1_s*.npz` |
| 5 | Réglage de FedAvg, FedProx et SCAFFOLD : convergence exigée, puis choix sur les hôpitaux de réglage | `outputs/e0_<tâche>/fl_config.json` |
| 6 | FedAvg, FedProx, SCAFFOLD pour toutes les graines, et diagnostic de convergence | `fl_r1_s*.npz` |
| 7 | Prédictions sur D et sur le test | `preds_r1.parquet`, `oof_local_r1.parquet` |
| 8 | Métriques E0 : par hôpital (moyens et grands), regroupées par strate, pire hôpital | `e0_*.csv` |
| 9 | Critère G1 et comparaisons statistiques | `e0_g1.csv` |
| 10 | Fiche de résultats | `E0_RESULTS.md` |

**Principes**
* Données chargées uniquement par `tdtfl_data.TDTData` (notebook 02) : normalisation globale sur le train de la fédération, harmonisation des variables indisponibles. Le notebook vérifie que le protocole est bien la version figée.
* Le **test** des hôpitaux d'évaluation ne sert qu'aux métriques. Tous les choix (régularisation, hyperparamètres fédérés, méthode fédérée de référence) sont faits sur le train (plis internes) ou sur **D des hôpitaux de réglage**.
* Le modèle centralisé et les trois méthodes fédérées optimisent **le même objectif** (log-loss moyenne + pénalité L2 au même niveau). L'écart entre eux mesure donc seulement l'effet de l'hétérogénéité et de l'optimisation fédérée.
* Les modèles de chaque graine sont sauvegardés : une exécution interrompue reprend où elle s'était arrêtée (`RECOMPUTE = False`).

**Durée indicative** : quelques minutes par graine. Faites un premier essai avec `SEEDS = [0]`, puis relancez avec les 10 graines : les modèles déjà calculés sont réutilisés.

In [1]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION
# ============================================================
import json, sys, time, importlib, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

# ---------- Chemins (les mêmes que dans le notebook 02) ----------
PROJECT_DIR = Path("../..")
TDT_DIR = PROJECT_DIR / "data_tdtfl"
SRC_DIR = PROJECT_DIR / "src"
OUT_DIR = PROJECT_DIR / "outputs"

# ---------- Expérience ----------
TASK = "mortality"                    # "mortality" (tâche principale) ou "aki" (tâche secondaire)
# SEEDS = [0]              # premier essai conseillé : SEEDS = [0]
SEEDS = list(range(10))               # premier essai conseillé : SEEDS = [0]
EXPECTED_PROTOCOL_HASH = "0f91266774f979f1"   # protocole figé, version 1.1
RECOMPUTE = False                     # True : réentraîne même si les modèles d'une graine existent
N_JOBS = -1                           # cœurs utilisés pour les modèles locaux (-1 = tous)

# ---------- Modèles ----------
C_GRID = (0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)   # régularisation L2 (C de scikit-learn)
TUNING_SEED = 0                       # graine utilisée pour régler les méthodes fédérées
FL_ROUNDS = 400
GAP_TOL = 1e-3                        # convergence exigée : écart d'objectif avec le centralisé <= GAP_TOL
FL_GRID = [{"local_steps": k, "step_scale": s} for k in (1, 5, 20) for s in (0.5, 1.0)]
FEDPROX_MU = (0.01, 0.1, 1.0)
EVAL_EVERY = 10

# ---------- Évaluation ----------
PER_HOSPITAL_STRATA = ("moyen", "grand")   # métriques par hôpital ; les petits sont regroupés par strate
G1_THRESHOLDS = {"auroc": 0.01, "brier": 0.002}   # PROPOSITION : à fixer avec votre directeur avant de lire E0
N_BOOT = 1000

OUT = OUT_DIR / f"e0_{TASK}"
MODELS = OUT_DIR / "models" / TASK
for d in (OUT, MODELS, SRC_DIR / "tdtfl"):
    d.mkdir(parents=True, exist_ok=True)
assert TASK in ("mortality", "aki")
print("Tâche :", TASK, "| graines :", SEEDS)
print("Sorties :", OUT.resolve(), "|", MODELS.resolve())

Tâche : mortality | graines : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Sorties : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\outputs\e0_mortality | C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\outputs\models\mortality


## Étape 1 — Modules `tdtfl.models` et `tdtfl.metrics`

Cette cellule écrit les deux modules que les notebooks suivants importeront (experts, décisions, E1 à E4).

* **Modèle local** : régression logistique L2 (scikit-learn, sans pondération des classes). La régularisation est choisie par les 5 plis internes du train (log-loss hors plis) ; les prédictions hors plis au C retenu sont gardées pour l'estimation croisée de E2. Un hôpital avec moins de 2 événements dans son train reçoit un modèle constant (cas possible pour l'AKI, pas pour la mortalité).
* **Méthodes fédérées** : FedAvg, FedProx et SCAFFOLD (option II), participation complète, K pas de descente de gradient locale par round, pas c / L_k propre à chaque client (L_k : constante de Lipschitz de sa log-loss). La pénalité vaut λ = 1 / (C·N), avec le C du modèle centralisé : les quatre modèles globaux optimisent le même objectif.
* **Métriques** : AUROC, AUPRC, Brier, intercept de calibration globale, pente de calibration, ECE (10 classes de même effectif), bootstrap par grappes d'hôpitaux.

In [2]:
# ============================================================
# ÉTAPE 1 — MODULES src/tdtfl/models.py ET src/tdtfl/metrics.py
# ============================================================
MODELS_SRC = r'''
"""
tdtfl.models — modèles de TDT-FL (protocole 1.1).

Généré par notebooks/03_modeles_E0.ipynb (étape 1) : modifier le notebook, pas ce fichier.

- Régression logistique L2 locale : force de régularisation C choisie par les plis internes du train
  (log-loss hors plis) ; les prédictions hors plis au C retenu sont conservées (estimation croisée de E2).
- FedAvg, FedProx et SCAFFOLD pour la même régression logistique : participation complète, K pas de
  descente de gradient locale par round, pas local c / L_k (L_k : constante de Lipschitz du client).
  Objectif commun : log-loss moyenne pondérée par les effectifs + (λ/2)·||w||², avec λ = 1 / (C·N) ;
  c'est exactement l'objectif de scikit-learn pour le modèle centralisé au même C.
"""
import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression

C_GRID = (0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)
MIN_CLASS = 2          # moins de 2 événements (ou non-événements) : modèle constant
MAX_ITER = 2000
EPS = 1e-6


def sigmoid(z):
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, dtype=np.float64)))


def logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def logloss(y, p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    y = np.asarray(y, dtype=np.float64)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


class Linear:
    """Modèle logistique : p = sigmoid(X w + b)."""

    def __init__(self, w, b, **meta):
        self.w = np.asarray(w, dtype=np.float64)
        self.b = float(b)
        self.meta = meta

    def predict(self, X):
        return sigmoid(np.asarray(X, dtype=np.float64) @ self.w + self.b)

    @property
    def theta(self):
        return np.r_[self.w, self.b]


def constant_model(y, n_features):
    prev = (float(np.sum(y)) + 0.5) / (len(y) + 1.0)          # prévalence lissée
    return Linear(np.zeros(n_features), logit(prev), kind="constant")


def _too_few(y):
    s = float(np.sum(y))
    return min(s, len(y) - s) < MIN_CLASS


def refit(X, y, C):
    """Régression logistique L2 à C fixé (modèle constant si une classe est trop rare)."""
    if _too_few(y):
        return constant_model(y, X.shape[1])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        est = LogisticRegression(C=C, max_iter=MAX_ITER).fit(X, y.astype(int))
    n_warn = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    return Linear(est.coef_[0].copy(), est.intercept_[0], kind="logistic", C=float(C), convergence_warnings=n_warn)


def fit_local(X, y, folds, c_grid=C_GRID):
    """Choisit C par les plis internes, puis réentraîne sur tout le train.
    Renvoie (modèle, prédictions hors plis au C retenu, log-loss hors plis par C)."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    folds = np.asarray(folds)
    n = len(y)
    if _too_few(y):
        m = constant_model(y, X.shape[1])
        return m, np.full(n, sigmoid(m.b)), [np.nan] * len(c_grid)
    oof = np.full((len(c_grid), n), np.nan)
    n_warn = 0
    for k in np.unique(folds[folds >= 0]):
        va = folds == k
        tr = ~va
        if _too_few(y[tr]):
            oof[:, va] = (y[tr].sum() + 0.5) / (tr.sum() + 1.0)
            continue
        est = LogisticRegression(C=c_grid[0], max_iter=MAX_ITER, warm_start=True)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            for ci, c in enumerate(c_grid):
                est.set_params(C=c)
                est.fit(X[tr], y[tr].astype(int))
                oof[ci, va] = est.predict_proba(X[va])[:, 1]
        n_warn += sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    losses = [logloss(y, o) for o in oof]
    best = int(np.nanargmin(losses))
    model = refit(X, y, c_grid[best])
    model.meta["cv_convergence_warnings"] = n_warn
    return model, oof[best], losses


# ----------------------------------------------------------------------------- fédéré
class Client:
    """Données d'entraînement d'un hôpital pour l'apprentissage fédéré."""

    def __init__(self, X, y, seed=0):
        self.X = np.asarray(X, dtype=np.float64)
        self.y = np.asarray(y, dtype=np.float64)
        self.n = len(self.y)
        self.L = lipschitz(self.X, seed=seed)


def lipschitz(X, n_iter=30, seed=0):
    """0,25 × plus grande valeur propre de [X 1]ᵀ[X 1] / n (puissance itérée)."""
    rng = np.random.default_rng(seed)
    n, p = X.shape
    v = rng.standard_normal(p + 1)
    v /= np.linalg.norm(v)
    lam = 1.0
    for _ in range(n_iter):
        u = X @ v[:-1] + v[-1]
        w = np.r_[X.T @ u, u.sum()] / n
        lam = float(np.linalg.norm(w))
        if lam == 0:
            return 0.25
        v = w / lam
    return 0.25 * lam


def _grad(theta, X, y, lam):
    r = sigmoid(X @ theta[:-1] + theta[-1]) - y
    g = np.empty_like(theta)
    g[:-1] = X.T @ r / len(y) + lam * theta[:-1]
    g[-1] = r.mean()
    return g


def objective(theta, clients, lam):
    """Log-loss moyenne pondérée par les effectifs + (λ/2)||w||² (objectif centralisé)."""
    N = sum(c.n for c in clients)
    loss = sum(c.n * logloss(c.y, sigmoid(c.X @ theta[:-1] + theta[-1])) for c in clients) / N
    return loss + 0.5 * lam * float(theta[:-1] @ theta[:-1])


def federated(clients, method="fedavg", rounds=100, local_steps=5, step_scale=1.0, lam=1e-4, mu=0.0,
              eval_fn=None, eval_every=10, theta0=None):
    """FedAvg, FedProx ou SCAFFOLD (option II) avec participation complète.
    Renvoie (modèle, historique [(round, eval_fn(theta))])."""
    if method not in ("fedavg", "fedprox", "scaffold"):
        raise ValueError(method)
    p = clients[0].X.shape[1]
    N = float(sum(c.n for c in clients))
    wts = [c.n / N for c in clients]
    theta = np.zeros(p + 1) if theta0 is None else np.array(theta0, dtype=np.float64)
    if method == "scaffold":
        c_glob = np.zeros(p + 1)
        c_loc = [np.zeros(p + 1) for _ in clients]
    history = []
    for r in range(1, rounds + 1):
        new = np.zeros_like(theta)
        dc = np.zeros_like(theta)
        for k, cl in enumerate(clients):
            eta = step_scale / (cl.L + lam + (mu if method == "fedprox" else 0.0))
            th = theta.copy()
            for _ in range(local_steps):
                g = _grad(th, cl.X, cl.y, lam)
                if method == "fedprox":
                    g += mu * (th - theta)
                elif method == "scaffold":
                    g += c_glob - c_loc[k]
                th -= eta * g
            if method == "scaffold":
                c_new = c_loc[k] - c_glob + (theta - th) / (local_steps * eta)
                dc += wts[k] * (c_new - c_loc[k])
                c_loc[k] = c_new
            new += wts[k] * th
        theta = new
        if method == "scaffold":
            c_glob = c_glob + dc
        if not np.all(np.isfinite(theta)):
            return None, history + [(r, np.inf)]
        if eval_fn is not None and (r % eval_every == 0 or r == rounds):
            history.append((r, float(eval_fn(theta))))
    model = Linear(theta[:-1], theta[-1], kind=method, rounds=rounds, local_steps=local_steps,
                   step_scale=step_scale, lam=lam, mu=mu)
    return model, history
'''

METRICS_SRC = r'''
"""
tdtfl.metrics — métriques de TDT-FL (protocole 1.1).

Généré par notebooks/03_modeles_E0.ipynb (étape 1) : modifier le notebook, pas ce fichier.
Discrimination (AUROC, AUPRC), calibration (Brier, intercept de calibration globale, pente, ECE à classes de
même effectif), bootstrap par grappes d'hôpitaux.
"""
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

EPS = 1e-6


def _two_classes(y):
    s = float(np.sum(y))
    return 0 < s < len(y)


def auroc(y, p):
    return float(roc_auc_score(y, p)) if _two_classes(y) else np.nan


def auprc(y, p):
    return float(average_precision_score(y, p)) if _two_classes(y) else np.nan


def brier(y, p):
    return float(np.mean((np.asarray(p, float) - np.asarray(y, float)) ** 2))


def _logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def _irls(y, Z, offset, iters=100):
    beta = np.zeros(Z.shape[1])
    for _ in range(iters):
        eta = Z @ beta + offset
        mu = 0.5 * (1.0 + np.tanh(0.5 * eta))
        W = mu * (1 - mu) + 1e-10
        H = (Z * W[:, None]).T @ Z + 1e-8 * np.eye(Z.shape[1])
        step = np.linalg.solve(H, Z.T @ (y - mu))
        beta += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return beta


def calibration(y, p):
    """(intercept de calibration globale, pente de calibration) ; idéal : (0, 1)."""
    y = np.asarray(y, dtype=np.float64)
    if not _two_classes(y):
        return np.nan, np.nan
    lp = _logit(p)
    a = _irls(y, np.ones((len(y), 1)), lp)[0]
    slope = _irls(y, np.column_stack([np.ones(len(y)), lp]), np.zeros(len(y)))[1]
    return float(a), float(slope)


def ece(y, p, n_bins=10):
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    order = np.argsort(p, kind="stable")
    out = 0.0
    for b in np.array_split(order, n_bins):
        if len(b):
            out += len(b) / len(y) * abs(y[b].mean() - p[b].mean())
    return float(out)


def all_metrics(y, p):
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    a, s = calibration(y, p)
    return {"n": int(len(y)), "events": int(y.sum()), "auroc": auroc(y, p), "auprc": auprc(y, p),
            "brier": brier(y, p), "cal_intercept": a, "cal_slope": s, "ece": ece(y, p)}


def cluster_bootstrap(groups, fn, n_boot=1000, seed=0):
    """Rééchantillonne les grappes (hôpitaux) avec remise ; fn(liste d'identifiants) -> statistique."""
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    return np.array([fn(rng.choice(groups, size=len(groups), replace=True)) for _ in range(n_boot)])
'''

pkg = SRC_DIR / "tdtfl"
(pkg / "__init__.py").write_text('"""TDT-FL : modules générés par les notebooks."""\n', encoding="utf-8")
(pkg / "models.py").write_text(MODELS_SRC.strip() + "\n", encoding="utf-8")
(pkg / "metrics.py").write_text(METRICS_SRC.strip() + "\n", encoding="utf-8")
print("Écrits :", sorted(p.name for p in pkg.glob("*.py")))

Écrits : ['__init__.py', 'metrics.py', 'models.py']


## Étape 2 — Données

Vérifie que le protocole est la version figée, définit la tâche et prépare le chargement d'une graine : pour chaque hôpital, la matrice encodée (normalisation globale R1 de la graine), le label, les parties (train, D, test) et les plis internes.

Pour l'AKI, seuls les séjours où l'AKI est évaluable (créatinine dosée aux deux périodes) sont utilisés, à l'entraînement comme à l'évaluation.

In [3]:
# ============================================================
# ÉTAPE 2 — DONNÉES : PROTOCOLE, TÂCHE, CHARGEMENT PAR GRAINE
# ============================================================
sys.path.insert(0, str(SRC_DIR.resolve()))
import tdtfl_data
import tdtfl.models as M
import tdtfl.metrics as MET
for mod in (tdtfl_data, M, MET):
    importlib.reload(mod)

data = tdtfl_data.TDTData(TDT_DIR)
if data.protocol["hash"] != EXPECTED_PROTOCOL_HASH:
    raise RuntimeError(f"Protocole {data.protocol['hash']} au lieu de {EXPECTED_PROTOCOL_HASH} : "
                       "ce n'est pas la version figée.")
R1 = tdtfl_data.R1
H = data.hospitals
HOSP = data.federation("r1")
EVAL = data.targets("r1", role="eval")
TUNE = data.targets("r1", role="tune")
STRATA = ["petit", "moyen", "grand"]
STRATUM = H["stratum"].to_dict()
FL_METHODS = ["fedavg", "fedprox", "scaffold"]
MODEL_NAMES = ["local", "central"] + FL_METHODS + (["apache"] if TASK == "mortality" else [])

def task_xy(d):
    """Label de la tâche et séjours utilisables."""
    if TASK == "mortality":
        return d["y_mortality"].astype(np.float64), np.ones(len(d["stay_id"]), dtype=bool)
    keep = d["aki_ascertained"].astype(bool)
    return np.nan_to_num(d["y_aki"].astype(np.float64)), keep

def load_seed(seed):
    """Données de tous les hôpitaux pour une graine (normalisation globale R1 de cette graine)."""
    norm = data.normalizer("r1", seed)
    out = {}
    for h in HOSP:
        d = data.matrix(h, "r1", seed, norm)
        y, keep = task_xy(d)
        out[h] = {"X": d["X"], "y": y, "keep": keep, "split": d["split"], "inner": d["inner_fold"],
                  "stay_id": d["stay_id"], "phenotype": d["phenotype"],
                  "apache": d.get("apache_pred_mortality", np.full(len(y), np.nan)).astype(np.float64)}
    return out

def part(d, name):
    return (d["split"] == R1[name]) & d["keep"]

D0 = load_seed(SEEDS[0])
cnt = pd.DataFrame([{"hospitalid": h, "stratum": STRATUM[h], "role": H.loc[h, "role"],
                     **{f"{p}_n": int(part(D0[h], p).sum()) for p in R1},
                     **{f"{p}_ev": int(D0[h]["y"][part(D0[h], p)].sum()) for p in R1}} for h in HOSP])
print(f"Protocole {data.protocol['hash']} (version {data.protocol['version']}) ; {len(HOSP)} hôpitaux "
      f"({len(EVAL)} d'évaluation, {len(TUNE)} de réglage) ; {D0[HOSP[0]]['X'].shape[1]} variables")
print(f"Graine {SEEDS[0]} — séjours et événements moyens par hôpital ({TASK}) :")
display(cnt.groupby("stratum")[[c for c in cnt.columns if c.endswith(("_n", "_ev"))]].mean().round(1).reindex(STRATA))

Protocole 0f91266774f979f1 (version 1.1) ; 101 hôpitaux (81 d'évaluation, 20 de réglage) ; 225 variables
Graine 0 — séjours et événements moyens par hôpital (mortality) :


,train_n,D_n,test_n,train_ev,D_ev,test_ev
stratum,,,,,,
petit,247.3,82.3,82.4,20.6,6.8,6.7
moyen,523.5,174.5,174.5,51.1,17.0,16.8
grand,1329.6,443.2,443.4,136.8,45.6,45.8


## Étape 3 — Modèles locaux

Un modèle par hôpital et par graine, entraîné sur son seul train. Les hôpitaux sont traités en parallèle. Le tableau final montre la régularisation retenue : si beaucoup d'hôpitaux tombent au bord de la grille, élargissez `C_GRID`.

In [4]:
# ============================================================
# ÉTAPE 3 — MODÈLES LOCAUX (RÉGRESSION LOGISTIQUE L2) + PRÉDICTIONS HORS PLIS
# ============================================================
from joblib import Parallel, delayed

def _fit_local_job(h, X, y, folds):
    m, oof, losses = M.fit_local(X, y, folds, C_GRID)
    return h, m, oof, losses

def local_path(seed):
    return MODELS / f"local_r1_s{seed}.npz"

for seed in SEEDS:
    if local_path(seed).exists() and not RECOMPUTE:
        print(f"graine {seed} : modèles locaux déjà calculés")
        continue
    t0 = time.time()
    D = D0 if seed == SEEDS[0] else load_seed(seed)
    jobs = [delayed(_fit_local_job)(h, D[h]["X"][part(D[h], "train")], D[h]["y"][part(D[h], "train")],
                                    D[h]["inner"][part(D[h], "train")]) for h in HOSP]
    res = Parallel(n_jobs=N_JOBS)(jobs)
    assert [r[0] for r in res] == HOSP
    np.savez_compressed(
        local_path(seed), hospital_ids=np.array(HOSP),
        W=np.vstack([m.w for _, m, _, _ in res]), b=np.array([m.b for _, m, _, _ in res]),
        C=np.array([m.meta.get("C", np.nan) for _, m, _, _ in res]),
        kind=np.array([m.meta["kind"] for _, m, _, _ in res]),
        cv_logloss=np.array([l for _, _, _, l in res], dtype=np.float64),
        convergence_warnings=np.array([m.meta.get("convergence_warnings", 0) + m.meta.get("cv_convergence_warnings", 0)
                                       for _, m, _, _ in res]),
        oof_stay_id=np.concatenate([D[h]["stay_id"][part(D[h], "train")] for h in HOSP]),
        oof_p=np.concatenate([oof for _, _, oof, _ in res]))
    print(f"graine {seed} : {len(res)} modèles locaux en {time.time() - t0:.0f} s")

rows = []
for seed in SEEDS:
    z = np.load(local_path(seed))
    rows += [{"seed": seed, "hospitalid": int(h), "stratum": STRATUM[int(h)], "C": c, "kind": k, "warn": int(w)}
             for h, c, k, w in zip(z["hospital_ids"], z["C"], z["kind"], z["convergence_warnings"])]
loc = pd.DataFrame(rows)
print("Régularisation retenue (part des modèles par valeur de C) :")
display(pd.crosstab(loc["stratum"], loc["C"], normalize="index").round(2).reindex(STRATA))
edge = loc["C"].isin([C_GRID[0], C_GRID[-1]]).mean()
print(f"Modèles constants : {(loc['kind'] == 'constant').sum()} ; au bord de la grille : {edge:.0%} ; "
      f"avertissements de convergence : {loc['warn'].sum()}")

graine 0 : modèles locaux déjà calculés
graine 1 : modèles locaux déjà calculés
graine 2 : modèles locaux déjà calculés
graine 3 : modèles locaux déjà calculés
graine 4 : modèles locaux déjà calculés
graine 5 : modèles locaux déjà calculés
graine 6 : modèles locaux déjà calculés
graine 7 : modèles locaux déjà calculés
graine 8 : modèles locaux déjà calculés
graine 9 : modèles locaux déjà calculés
Régularisation retenue (part des modèles par valeur de C) :


C,0.0001,0.0003,0.0010,0.0030,0.0100,0.0300,0.1000,0.3000
stratum,,,,,,,,
petit,0.01,0.0,0.01,0.06,0.43,0.44,0.05,0.0
moyen,0.00,0.0,0.00,0.04,0.64,0.31,0.01,0.0
grand,0.00,0.0,0.00,0.01,0.56,0.42,0.01,0.0


Modèles constants : 0 ; au bord de la grille : 0% ; avertissements de convergence : 0


## Étape 4 — Modèle centralisé

Toutes les données d'entraînement réunies (borne supérieure : ce qu'on obtiendrait si les données pouvaient être mises en commun). La régularisation est choisie par les mêmes plis internes. Elle fixe aussi la pénalité λ des méthodes fédérées.

In [5]:
# ============================================================
# ÉTAPE 4 — MODÈLE CENTRALISÉ
# ============================================================
def central_path(seed):
    return MODELS / f"central_r1_s{seed}.npz"

def pooled_train(D):
    X = np.vstack([D[h]["X"][part(D[h], "train")] for h in HOSP])
    y = np.concatenate([D[h]["y"][part(D[h], "train")] for h in HOSP])
    f = np.concatenate([D[h]["inner"][part(D[h], "train")] for h in HOSP])
    return X, y, f

def get_central(seed, D=None):
    if not central_path(seed).exists() or RECOMPUTE:
        t0 = time.time()
        D = D if D is not None else load_seed(seed)
        X, y, f = pooled_train(D)
        m, _, losses = M.fit_local(X, y, f, C_GRID)
        np.savez_compressed(central_path(seed), theta=m.theta, C=m.meta["C"], lam=1.0 / (m.meta["C"] * len(y)),
                            n_train=len(y), cv_logloss=np.array(losses))
        print(f"graine {seed} : modèle centralisé en {time.time() - t0:.0f} s (C = {m.meta['C']})")
    z = np.load(central_path(seed))
    return {k: z[k] for k in z.files}

for seed in SEEDS:
    c = get_central(seed, D0 if seed == SEEDS[0] else None)
print(pd.DataFrame([{"seed": s, "C": float(get_central(s)["C"]), "lambda": float(get_central(s)["lam"]),
                     "n_train": int(get_central(s)["n_train"])} for s in SEEDS]).to_string(index=False))

graine 3 : modèle centralisé en 55 s (C = 0.1)
graine 4 : modèle centralisé en 58 s (C = 0.1)
graine 5 : modèle centralisé en 51 s (C = 0.1)
graine 6 : modèle centralisé en 35 s (C = 0.1)
graine 7 : modèle centralisé en 42 s (C = 0.1)
graine 8 : modèle centralisé en 33 s (C = 0.1)
graine 9 : modèle centralisé en 39 s (C = 0.1)
 seed   C   lambda  n_train
    0 0.1 0.000209    47930
    1 0.1 0.000209    47921
    2 0.1 0.000209    47922
    3 0.1 0.000209    47927
    4 0.1 0.000209    47925
    5 0.1 0.000209    47922
    6 0.1 0.000209    47921
    7 0.1 0.000209    47930
    8 0.1 0.000209    47931
    9 0.1 0.000209    47926


## Étape 5 — Réglage des méthodes fédérées (hôpitaux de réglage)

Sur la graine de réglage, chaque configuration est entraînée sur le train des 101 hôpitaux. Le choix se fait en deux temps :

1. **Convergence exigée** : l'objectif atteint doit être à moins de `GAP_TOL` (10⁻³) de celui du modèle centralisé, qui optimise exactement le même objectif. Sans cette règle, le réglage préférerait les configurations qui s'arrêtent tôt : l'arrêt précoce agit comme une régularisation cachée, et le modèle fédéré sort mal calibré (pente de calibration nettement supérieure à 1). La comparaison avec le local ne serait plus loyale.
2. Parmi les configurations qui ont convergé, la plus faible **log-loss sur D des hôpitaux de réglage** : ni le test ni les hôpitaux d'évaluation n'interviennent.

* FedAvg et SCAFFOLD : nombre de pas locaux K ∈ {1, 5, 20} × facteur de pas c ∈ {0,5 ; 1}.
* FedProx : μ ∈ {0,01 ; 0,1 ; 1}, avec le (K, c) retenu pour FedAvg.
* La **méthode fédérée de référence** (celle du critère G1 et du modèle global G des notebooks suivants) est celle qui a la plus faible log-loss.
* Si aucune configuration d'une méthode ne converge, celle qui s'approche le plus est retenue et un avertissement s'affiche : augmentez alors `FL_ROUNDS`.

In [ ]:
# ============================================================
# ÉTAPE 5 — RÉGLAGE DE FEDAVG, FEDPROX ET SCAFFOLD (HÔPITAUX DE RÉGLAGE)
# ============================================================
cfg_path = OUT / "fl_config.json"

def make_clients(D):
    return [M.Client(D[h]["X"][part(D[h], "train")], D[h]["y"][part(D[h], "train")], seed=i)
            for i, h in enumerate(HOSP)]

if cfg_path.exists() and not RECOMPUTE:
    fl_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    print("Réglage déjà fait :", fl_cfg["reference"], fl_cfg["methods"])
else:
    t0 = time.time()
    D = D0 if TUNING_SEED == SEEDS[0] else load_seed(TUNING_SEED)
    cen = get_central(TUNING_SEED, D)
    lam = float(cen["lam"])
    clients = make_clients(D)
    Xd = np.vstack([D[h]["X"][part(D[h], "D")] for h in TUNE])
    yd = np.concatenate([D[h]["y"][part(D[h], "D")] for h in TUNE])
    tune_loss = lambda th: M.logloss(yd, M.sigmoid(Xd @ th[:-1] + th[-1]))
    obj_central = M.objective(cen["theta"], clients, lam)
    runs = []
    def run(method, **kw):
        model, hist = M.federated(clients, method, rounds=FL_ROUNDS, lam=lam, eval_fn=tune_loss,
                                  eval_every=EVAL_EVERY, **kw)
        final = hist[-1][1] if (hist and model is not None) else np.inf
        gap = M.objective(model.theta, clients, lam) - obj_central if model is not None else np.inf
        runs.append({"method": method, **kw, "tune_logloss": final, "objective_gap": gap,
                     "converged": bool(gap <= GAP_TOL), "diverged": model is None,
                     "history": [round(v, 5) for _, v in hist]})
        print(f"  {method:9s} {kw} -> log-loss {final:.5f} ; écart d'objectif {gap:.1e}")
    def choose(rows):
        ok = [r for r in rows if r["converged"]]
        if not ok:
            print(f"  Attention : aucune configuration de {rows[0]['method']} n'a convergé ; "
                  "la plus proche est retenue (augmentez FL_ROUNDS).")
            return min(rows, key=lambda r: r["objective_gap"])
        return min(ok, key=lambda r: r["tune_logloss"])
    for g in FL_GRID:
        run("fedavg", **g)
        run("scaffold", **g)
    best_avg = choose([r for r in runs if r["method"] == "fedavg"])
    for mu in FEDPROX_MU:
        run("fedprox", local_steps=best_avg["local_steps"], step_scale=best_avg["step_scale"], mu=mu)
    tab = pd.DataFrame(runs)
    best = {m: pd.Series(choose([r for r in runs if r["method"] == m])) for m in FL_METHODS}
    methods = {m: {k: (float(v) if k in ("step_scale", "mu") else int(v)) for k, v in best[m].items()
                   if k in ("local_steps", "step_scale", "mu") and pd.notna(v)} for m in FL_METHODS}
    reference = min(FL_METHODS, key=lambda m: best[m]["tune_logloss"])
    lz = np.load(local_path(TUNING_SEED)) if local_path(TUNING_SEED).exists() else None
    p_loc = (np.concatenate([M.sigmoid(D[h]["X"][part(D[h], "D")] @ lz["W"][HOSP.index(h)] + lz["b"][HOSP.index(h)])
                             for h in TUNE]) if lz is not None else None)
    fl_cfg = {"tuning_seed": TUNING_SEED, "rounds": FL_ROUNDS, "lambda_tuning_seed": lam,
              "criterion": "convergence (écart d'objectif <= gap_tol), puis log-loss sur D des hôpitaux de réglage",
              "methods": methods, "reference": reference,
              "tune_logloss": {m: float(best[m]["tune_logloss"]) for m in FL_METHODS},
              "objective_gap": {m: float(best[m]["objective_gap"]) for m in FL_METHODS},
              "converged": {m: bool(best[m]["converged"]) for m in FL_METHODS}, "gap_tol": GAP_TOL,
              "tune_logloss_central": float(tune_loss(cen["theta"])),
              "tune_logloss_local": float(M.logloss(yd, p_loc)) if p_loc is not None else None,
              "created_at": datetime.now().isoformat(timespec="seconds")}
    cfg_path.write_text(json.dumps(fl_cfg, indent=2, ensure_ascii=False), encoding="utf-8")
    tab.drop(columns="history").to_csv(OUT / "fl_tuning.csv", index=False)
    pd.DataFrame([{"method": r["method"], "local_steps": r.get("local_steps"), "step_scale": r.get("step_scale"),
                   "mu": r.get("mu"), **{f"r{(i + 1) * EVAL_EVERY}": v for i, v in enumerate(r["history"])}}
                  for r in runs]).to_csv(OUT / "fl_tuning_curves.csv", index=False)
    print(f"Réglage en {time.time() - t0:.0f} s")

print("Méthode fédérée de référence :", fl_cfg["reference"])
display(pd.DataFrame(fl_cfg["methods"]).T.assign(tune_logloss=pd.Series(fl_cfg["tune_logloss"]),
                                                 objective_gap=pd.Series(fl_cfg["objective_gap"])))
print(f"Log-loss sur D des hôpitaux de réglage — centralisé : {fl_cfg['tune_logloss_central']:.5f} ; "
      f"modèles locaux : {fl_cfg['tune_logloss_local']:.5f}" if fl_cfg.get("tune_logloss_local") is not None else "")

  fedavg    {'local_steps': 1, 'step_scale': 0.5} -> log-loss 0.22422 ; écart d'objectif 1.2e-02
  scaffold  {'local_steps': 1, 'step_scale': 0.5} -> log-loss 0.22407 ; écart d'objectif 1.2e-02
  fedavg    {'local_steps': 1, 'step_scale': 1.0} -> log-loss 0.22190 ; écart d'objectif 8.3e-03
  scaffold  {'local_steps': 1, 'step_scale': 1.0} -> log-loss 0.22175 ; écart d'objectif 8.2e-03
  fedavg    {'local_steps': 5, 'step_scale': 0.5} -> log-loss 0.22014 ; écart d'objectif 4.8e-03
  scaffold  {'local_steps': 5, 'step_scale': 0.5} -> log-loss 0.21972 ; écart d'objectif 4.5e-03
  fedavg    {'local_steps': 5, 'step_scale': 1.0} -> log-loss 0.21948 ; écart d'objectif 3.0e-03
  scaffold  {'local_steps': 5, 'step_scale': 1.0} -> log-loss 0.21885 ; écart d'objectif 2.5e-03
  fedavg    {'local_steps': 20, 'step_scale': 0.5} -> log-loss 0.21946 ; écart d'objectif 1.9e-03
  scaffold  {'local_steps': 20, 'step_scale': 0.5} -> log-loss 0.21846 ; écart d'objectif 1.2e-03


## Étape 6 — Méthodes fédérées pour toutes les graines

Chaque méthode est entraînée avec sa configuration réglée et la pénalité λ de la graine. Deux diagnostics la comparent au modèle centralisé, qui optimise exactement le même objectif :

* l'**écart d'objectif** : de l'ordre de 10⁻³ ou moins, la méthode a pratiquement convergé. La régression logistique sur des résumés très corrélés (moyenne, minimum, maximum, dernière valeur d'une même variable) est mal conditionnée : la fin de la convergence est lente, mais elle porte sur des directions qui changent peu les prédictions ;
* l'**écart moyen des prédictions** sur D et le test : c'est lui qui compte pour E0. En dessous de 0,005 environ, le modèle fédéré et le centralisé prédisent la même chose.

In [7]:
# ============================================================
# ÉTAPE 6 — FEDAVG, FEDPROX ET SCAFFOLD POUR TOUTES LES GRAINES
# ============================================================
def fl_path(seed):
    return MODELS / f"fl_r1_s{seed}.npz"

for seed in SEEDS:
    if fl_path(seed).exists() and not RECOMPUTE:
        print(f"graine {seed} : méthodes fédérées déjà calculées")
        continue
    t0 = time.time()
    D = D0 if seed == SEEDS[0] else load_seed(seed)
    cen = get_central(seed, D)
    lam = float(cen["lam"])
    clients = make_clients(D)
    obj = lambda th: M.objective(th, clients, lam)
    obj_central = obj(cen["theta"])
    save = {"lam": lam, "objective_central": obj_central}
    for m in FL_METHODS:
        model, hist = M.federated(clients, m, rounds=FL_ROUNDS, lam=lam, eval_fn=obj, eval_every=EVAL_EVERY,
                                  **fl_cfg["methods"][m])
        if model is None:
            raise RuntimeError(f"graine {seed} : {m} a divergé ; réduisez step_scale dans FL_GRID")
        save[f"theta_{m}"] = model.theta
        save[f"gap_{m}"] = np.array([v - obj_central for _, v in hist])
        Xe = np.vstack([D[h]["X"][part(D[h], "D") | part(D[h], "test")] for h in HOSP]).astype(np.float64)
        save[f"pdiff_{m}"] = float(np.mean(np.abs(model.predict(Xe) - M.sigmoid(Xe @ cen["theta"][:-1] + cen["theta"][-1]))))
    np.savez_compressed(fl_path(seed), **save)
    print(f"graine {seed} : méthodes fédérées en {time.time() - t0:.0f} s")

conv = []
for seed in SEEDS:
    z = np.load(fl_path(seed))
    conv += [{"seed": seed, "method": m, "écart_final": float(z[f"gap_{m}"][-1]),
              "écart_mi_parcours": float(z[f"gap_{m}"][len(z[f"gap_{m}"]) // 2 - 1]),
              "écart_prédictions": float(z[f"pdiff_{m}"])} for m in FL_METHODS]
conv = pd.DataFrame(conv)
print("Écarts avec le modèle centralisé : objectif (log-loss pénalisée, train de la fédération) et "
      "prédictions (moyenne des écarts absolus, D et test) :")
display(conv.groupby("method")[["écart_mi_parcours", "écart_final", "écart_prédictions"]].agg(["mean", "max"])
        .map(lambda v: f"{v:.2e}"))

graine 0 : méthodes fédérées en 377 s
graine 1 : méthodes fédérées en 577 s
graine 2 : méthodes fédérées en 355 s
graine 4 : méthodes fédérées en 456 s
graine 5 : méthodes fédérées en 377 s
graine 6 : méthodes fédérées en 295 s
graine 7 : méthodes fédérées en 298 s
graine 8 : méthodes fédérées en 305 s
graine 9 : méthodes fédérées en 296 s
Écarts avec le modèle centralisé : objectif (log-loss pénalisée, train de la fédération) et prédictions (moyenne des écarts absolus, D et test) :


écart_mi_parcours           écart_final           écart_prédictions          
                      mean       max        mean       max              mean       max
method                                                                                
fedavg            2.39e-03  2.66e-03    1.58e-03  1.80e-03          1.03e-02  1.11e-02
fedprox           2.40e-03  2.67e-03    1.58e-03  1.79e-03          1.02e-02  1.11e-02
scaffold          1.09e-03  1.18e-03    3.92e-04  4.35e-04          4.10e-03  4.52e-03

## Étape 7 — Prédictions sur D et sur le test

Une ligne par séjour de D ou du test et par graine, avec la prédiction de chaque modèle. Pour la mortalité, la prédiction APACHE IVa est ajoutée comme référence clinique : elle n'est jamais une entrée.

`oof_local_r1.parquet` contient les prédictions hors plis du modèle local sur le train (estimation croisée de E2, notebook 05).

In [8]:
# ============================================================
# ÉTAPE 7 — PRÉDICTIONS SUR D ET SUR LE TEST
# ============================================================
frames, oofs = [], []
for seed in SEEDS:
    D = D0 if seed == SEEDS[0] else load_seed(seed)
    loc_z, cen, fl = np.load(local_path(seed)), get_central(seed), np.load(fl_path(seed))
    thetas = {"central": cen["theta"], **{m: fl[f"theta_{m}"] for m in FL_METHODS}}
    for i, h in enumerate(HOSP):
        d = D[h]
        rows = ((d["split"] == R1["D"]) | (d["split"] == R1["test"])) & d["keep"]
        X = d["X"][rows].astype(np.float64)
        rec = {"stay_id": d["stay_id"][rows], "seed": np.int8(seed), "hospitalid": np.int32(h),
               "split": d["split"][rows].astype(np.int8), "phenotype": d["phenotype"][rows], "y": d["y"][rows],
               "p_local": M.sigmoid(X @ loc_z["W"][i] + loc_z["b"][i])}
        for name, th in thetas.items():
            rec[f"p_{name}"] = M.sigmoid(X @ th[:-1] + th[-1])
        if TASK == "mortality":
            rec["p_apache"] = d["apache"][rows]
        frames.append(pd.DataFrame(rec))
    oofs.append(pd.DataFrame({"stay_id": loc_z["oof_stay_id"], "seed": np.int8(seed), "p_local_oof": loc_z["oof_p"]}))
P = pd.concat(frames, ignore_index=True)
for c in [c for c in P.columns if c.startswith("p_")]:
    P[c] = P[c].astype(np.float32)
P.to_parquet(OUT / "preds_r1.parquet", index=False)
pd.concat(oofs, ignore_index=True).to_parquet(OUT / "oof_local_r1.parquet", index=False)
print(f"{len(P)} prédictions (D et test, {len(SEEDS)} graine(s)) ; "
      f"APACHE disponible : {P['p_apache'].notna().mean():.1%}" if TASK == "mortality" else f"{len(P)} prédictions")

319475 prédictions (D et test, 10 graine(s)) ; APACHE disponible : 87.9%


## Étape 8 — Métriques E0

Sur le **test des hôpitaux d'évaluation** uniquement.

* **Par hôpital** pour les strates moyenne et grande : moyenne sur les hôpitaux (médiane pour l'intercept et la pente de calibration, très instables hôpital par hôpital), puis sur les graines ; et **pire hôpital** (AUROC la plus basse, Brier le plus haut) de chaque graine.
* **Regroupées par strate** : les prédictions de tous les hôpitaux d'évaluation d'une strate sont réunies avant le calcul. C'est la seule mesure fiable pour les petits hôpitaux (environ 7 décès dans le test de chacun).
* APACHE IVa est évalué sur les séjours où sa prédiction existe.

In [9]:
# ============================================================
# ÉTAPE 8 — MÉTRIQUES E0 (TEST DES HÔPITAUX D'ÉVALUATION)
# ============================================================
P = pd.read_parquet(OUT / "preds_r1.parquet")
T = P[(P["split"] == R1["test"]) & P["hospitalid"].isin(EVAL)].copy()
T["stratum"] = T["hospitalid"].map(STRATUM)
METRICS = ["auroc", "auprc", "brier", "cal_intercept", "cal_slope", "ece"]

def metrics_of(g, name):
    if name == "apache":
        g = g[g["p_apache"].notna()]
    return MET.all_metrics(g["y"].to_numpy(), g[f"p_{name}"].to_numpy())

per_h = pd.DataFrame([{"seed": s, "hospitalid": h, "stratum": STRATUM[h], "model": m, **metrics_of(g, m)}
                      for (s, h), g in T.groupby(["seed", "hospitalid"]) for m in MODEL_NAMES])
pooled = pd.DataFrame([{"seed": s, "stratum": st, "model": m, **metrics_of(g, m)}
                       for (s, st), g in T.groupby(["seed", "stratum"]) for m in MODEL_NAMES])
per_h.to_csv(OUT / "e0_par_hopital.csv", index=False)
pooled.to_csv(OUT / "e0_par_strate.csv", index=False)

def fmt(df):
    return df.reindex(index=MODEL_NAMES).round(4)

print("Regroupé par strate (moyenne sur les graines) :")
summary_pooled = pooled.groupby(["stratum", "model"])[METRICS].mean()
for st in STRATA:
    print(f"— {st}")
    display(fmt(summary_pooled.loc[st]))

ph = per_h[per_h["stratum"].isin(PER_HOSPITAL_STRATA)]
AGG = {m: ("median" if m.startswith("cal_") else "mean") for m in METRICS}   # pentes par hôpital : médiane (instables)
summary_hosp = ph.groupby(["seed", "stratum", "model"]).agg(AGG).groupby(["stratum", "model"]).mean()
worst = (ph.groupby(["seed", "stratum", "model"]).agg(pire_auroc=("auroc", "min"), pire_brier=("brier", "max"))
         .groupby(["stratum", "model"]).mean())
print("Par hôpital, strates moyenne et grande (moyenne sur les hôpitaux, médiane pour la calibration, "
      "puis moyenne sur les graines) et pire hôpital :")
for st in PER_HOSPITAL_STRATA:
    print(f"— {st}")
    display(fmt(summary_hosp.loc[st].join(worst.loc[st])))

C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Stryve\anaconda3\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount

Regroupé par strate (moyenne sur les graines) :
— petit


,auroc,auprc,brier,cal_intercept,cal_slope,ece
model,,,,,,
local,0.7956,0.3378,0.0625,0.0643,0.9660,0.0122
central,0.8648,0.4485,0.0562,-0.1436,0.9971,0.0112
fedavg,0.8633,0.4387,0.0568,-0.0768,0.9595,0.0104
fedprox,0.8632,0.4387,0.0568,-0.0779,0.9599,0.0103
scaffold,0.8638,0.4475,0.0562,-0.1427,1.0056,0.0108
apache,0.8360,0.3726,0.0631,-0.5251,0.9222,0.0352


— moyen


,auroc,auprc,brier,cal_intercept,cal_slope,ece
model,,,,,,
local,0.8373,0.4247,0.0709,0.0078,1.0459,0.0121
central,0.8731,0.5004,0.0657,0.0367,0.9971,0.0106
fedavg,0.8727,0.4959,0.0663,0.1021,0.9713,0.0125
fedprox,0.8727,0.4959,0.0662,0.1010,0.9716,0.0125
scaffold,0.8728,0.4991,0.0658,0.0379,1.0049,0.0113
apache,0.8416,0.4112,0.0752,-0.4944,0.9168,0.0385


— grand


,auroc,auprc,brier,cal_intercept,cal_slope,ece
model,,,,,,
local,0.8510,0.4742,0.0728,0.0051,0.9967,0.0121
central,0.8706,0.5166,0.0695,0.0663,0.9745,0.0100
fedavg,0.8712,0.5147,0.0699,0.1390,0.9614,0.0122
fedprox,0.8712,0.5147,0.0699,0.1379,0.9618,0.0121
scaffold,0.8703,0.5161,0.0695,0.0652,0.9829,0.0100
apache,0.8347,0.4450,0.0793,-0.3928,0.9165,0.0323


Par hôpital, strates moyenne et grande (moyenne sur les hôpitaux, médiane pour la calibration, puis moyenne sur les graines) et pire hôpital :
— moyen


,auroc,auprc,brier,cal_intercept,cal_slope,ece,pire_auroc,pire_brier
model,,,,,,,,
local,0.8285,0.4607,0.0774,0.0129,1.1141,0.0528,0.7066,0.1274
central,0.8682,0.5361,0.0714,0.1086,1.0513,0.0492,0.7617,0.1181
fedavg,0.8688,0.5355,0.0721,0.1749,1.0406,0.0513,0.7590,0.1204
fedprox,0.8688,0.5354,0.0721,0.1737,1.0410,0.0514,0.7588,0.1204
scaffold,0.8681,0.5348,0.0715,0.1143,1.0640,0.0497,0.7581,0.1186
apache,0.8393,0.4732,0.0818,-0.4548,0.9823,0.0683,0.7041,0.1455


— grand


,auroc,auprc,brier,cal_intercept,cal_slope,ece,pire_auroc,pire_brier
model,,,,,,,,
local,0.8490,0.4901,0.0739,0.0097,1.0315,0.0309,0.7893,0.0923
central,0.8692,0.5288,0.0705,0.0729,0.9995,0.0289,0.8221,0.0878
fedavg,0.8695,0.5295,0.0708,0.1296,0.9773,0.0301,0.8196,0.0885
fedprox,0.8694,0.5295,0.0708,0.1283,0.9776,0.0301,0.8197,0.0885
scaffold,0.8691,0.5295,0.0705,0.0662,1.0113,0.0298,0.8214,0.0878
apache,0.8347,0.4634,0.0811,-0.3824,0.9160,0.0489,0.7730,0.0976


## Étape 9 — Critère G1 et comparaisons

**G1 (marge)** : dans la strate des petits hôpitaux, le modèle local fait-il nettement moins bien que la méthode fédérée de référence ? Les seuils proposés (0,01 d'AUROC ou 0,002 de Brier) **doivent être fixés avec votre directeur avant de lire ce tableau**.

* Différences regroupées par strate (moyenne sur les graines), avec un intervalle de confiance à 95 % par bootstrap des hôpitaux.
* Strates moyenne et grande : différences par hôpital, test des rangs signés de Wilcoxon.
* Même calcul pour « fédéré contre centralisé » : le fédéré atteint-il la borne supérieure ?

In [10]:
# ============================================================
# ÉTAPE 9 — CRITÈRE G1 ET COMPARAISONS STATISTIQUES
# ============================================================
from scipy.stats import wilcoxon

REF = fl_cfg["reference"]
by = {(s, h): (g["y"].to_numpy(), {m: g[f"p_{m}"].to_numpy() for m in ("local", "central", REF)})
      for (s, h), g in T.groupby(["seed", "hospitalid"])}
seeds_run = sorted(int(s) for s in T["seed"].unique())

def pooled_delta(hs, a, b):
    """(AUROC_b − AUROC_a, Brier_a − Brier_b) regroupés sur les hôpitaux hs, moyenne sur les graines.
    Positif = b meilleur que a."""
    out = []
    for s in seeds_run:
        y = np.concatenate([by[(s, h)][0] for h in hs])
        pa = np.concatenate([by[(s, h)][1][a] for h in hs])
        pb = np.concatenate([by[(s, h)][1][b] for h in hs])
        out.append((MET.auroc(y, pb) - MET.auroc(y, pa), MET.brier(y, pa) - MET.brier(y, pb)))
    return np.nanmean(np.array(out), axis=0)

rows = []
for a, b, label in [("local", REF, f"{REF} − local"), (REF, "central", f"centralisé − {REF}"), ("local", "central", "centralisé − local")]:
    for st in STRATA:
        hs = [h for h in EVAL if STRATUM[h] == st]
        point = pooled_delta(hs, a, b)
        boot = MET.cluster_bootstrap(hs, lambda s: pooled_delta(list(s), a, b), n_boot=N_BOOT, seed=1)
        rec = {"comparaison": label, "strate": st, "hopitaux": len(hs),
               "gain_AUROC": point[0], "IC95_AUROC": f"[{np.percentile(boot[:, 0], 2.5):.4f} ; {np.percentile(boot[:, 0], 97.5):.4f}]",
               "gain_Brier": point[1], "IC95_Brier": f"[{np.percentile(boot[:, 1], 2.5):.4f} ; {np.percentile(boot[:, 1], 97.5):.4f}]"}
        if st in PER_HOSPITAL_STRATA:
            d = per_h[per_h["stratum"] == st].pivot_table(index="hospitalid", columns="model", values=["auroc", "brier"])
            da, db = d[("auroc", b)] - d[("auroc", a)], d[("brier", a)] - d[("brier", b)]
            rec["p_Wilcoxon_AUROC"] = wilcoxon(da).pvalue if len(da) > 5 and np.any(da != 0) else np.nan
            rec["p_Wilcoxon_Brier"] = wilcoxon(db).pvalue if len(db) > 5 and np.any(db != 0) else np.nan
        rows.append(rec)
g1 = pd.DataFrame(rows)
g1.to_csv(OUT / "e0_g1.csv", index=False)
display(g1.round(4))

small = g1[(g1["comparaison"] == f"{REF} − local") & (g1["strate"] == "petit")].iloc[0]
met = (small["gain_AUROC"] >= G1_THRESHOLDS["auroc"]) or (small["gain_Brier"] >= G1_THRESHOLDS["brier"])
G1_VERDICT = (f"G1 ({TASK}) : dans les petits hôpitaux, {REF} gagne {small['gain_AUROC']:+.4f} d'AUROC "
              f"{small['IC95_AUROC']} et {small['gain_Brier']:+.4f} de Brier {small['IC95_Brier']} sur le modèle local. "
              f"Seuils proposés {G1_THRESHOLDS} : {'ATTEINT' if met else 'NON ATTEINT'}.")
print(G1_VERDICT)

,comparaison,strate,hopitaux,gain_AUROC,IC95_AUROC,gain_Brier,IC95_Brier,p_Wilcoxon_AUROC,p_Wilcoxon_Brier
0,scaffold − local,petit,40,0.0682,[0.0530 ; 0.0832],0.0063,[0.0044 ; 0.0083],NaN,NaN
1,scaffold − local,moyen,32,0.0355,[0.0274 ; 0.0434],0.0051,[0.0039 ; 0.0066],0.0000,0.0000
2,scaffold − local,grand,9,0.0194,[0.0132 ; 0.0266],0.0033,[0.0026 ; 0.0042],0.0039,0.0039
3,centralisé − scaffold,petit,40,0.0010,[-0.0000 ; 0.0019],0.0001,[-0.0001 ; 0.0002],NaN,NaN
4,centralisé − scaffold,moyen,32,0.0003,[-0.0003 ; 0.0010],0.0001,[0.0000 ; 0.0002],0.8609,0.0228
5,centralisé − scaffold,grand,9,0.0003,[-0.0005 ; 0.0009],0.0000,[-0.0001 ; 0.0002],0.8203,0.6523
6,centralisé − local,petit,40,0.0692,[0.0535 ; 0.0840],0.0064,[0.0044 ; 0.0084],NaN,NaN
7,centralisé − local,moyen,32,0.0358,[0.0276 ; 0.0437],0.0052,[0.0039 ; 0.0067],0.0000,0.0000
8,centralisé − local,grand,9,0.0196,[0.0131 ; 0.0272],0.0033,[0.0026 ; 0.0042],0.0039,0.0039


G1 (mortality) : dans les petits hôpitaux, scaffold gagne +0.0682 d'AUROC [0.0530 ; 0.0832] et +0.0063 de Brier [0.0044 ; 0.0083] sur le modèle local. Seuils proposés {'auroc': 0.01, 'brier': 0.002} : ATTEINT.


## Étape 10 — Fiche de résultats

`E0_RESULTS.md` réunit la configuration, le réglage fédéré, la convergence, les métriques et le critère G1. C'est le fichier à m'envoyer.

In [11]:
# ============================================================
# ÉTAPE 10 — FICHE DE RÉSULTATS E0
# ============================================================
def md_table(df, index=True, digits=4):
    d = df.reset_index() if index else df
    fmt = lambda v: "" if (isinstance(v, float) and np.isnan(v)) else (f"{v:.{digits}f}" if isinstance(v, (float, np.floating)) else str(v))
    lines = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * len(d.columns)]
    lines += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join(lines)

parts = [f"# E0 — Marge du modèle local face au fédéré ({TASK})",
         f"Généré le {datetime.now():%Y-%m-%d %H:%M} par `notebooks/03_modeles_E0.ipynb`. "
         f"Protocole `{data.protocol['hash']}` (version {data.protocol['version']}) ; graines : {seeds_run}.",
         "## Configuration",
         "\n".join([f"- Local : régression logistique L2, C ∈ {list(C_GRID)} choisi par 5 plis internes (log-loss).",
                    f"- Fédéré : {FL_ROUNDS} rounds, participation complète, graine de réglage {TUNING_SEED}.",
                    f"- Choix : convergence exigée (écart d'objectif ≤ {GAP_TOL:g}), puis log-loss sur D des hôpitaux de réglage.",
                    f"- Méthode de référence : **{REF}** ; configurations : `{json.dumps(fl_cfg['methods'])}`."]),
         "## Réglage fédéré (graine de réglage)",
         md_table(pd.DataFrame({"tune_logloss": fl_cfg["tune_logloss"], "objective_gap": fl_cfg["objective_gap"],
                                "converged": fl_cfg["converged"]}).rename_axis("méthode"), digits=5),
         f"Log-loss sur D des hôpitaux de réglage — centralisé : {fl_cfg['tune_logloss_central']:.5f} ; modèles locaux : "
         + (f"{fl_cfg['tune_logloss_local']:.5f}." if fl_cfg.get("tune_logloss_local") is not None else "non calculée."),
         "## Convergence (écarts avec le centralisé en fin d'entraînement, moyenne et maximum sur les graines)",
         md_table(conv.groupby("method")[["écart_final", "écart_prédictions"]].agg(["mean", "max"])
                  .set_axis(["objectif_moy", "objectif_max", "prédictions_moy", "prédictions_max"], axis=1)
                  .rename_axis("méthode"), digits=5),
         "## Métriques regroupées par strate (test des hôpitaux d'évaluation, moyenne sur les graines)"]
for st in STRATA:
    parts += [f"### {st}", md_table(fmt(summary_pooled.loc[st]).rename_axis("modèle"))]
parts.append("## Par hôpital (strates moyenne et grande) et pire hôpital")
for st in PER_HOSPITAL_STRATA:
    parts += [f"### {st}", md_table(fmt(summary_hosp.loc[st].join(worst.loc[st])).rename_axis("modèle"))]
parts += ["## Critère G1 et comparaisons", md_table(g1, index=False), "", f"**{G1_VERDICT}**",
          "", "Gains positifs = le second modèle est meilleur. Intervalles : bootstrap des hôpitaux "
          f"({N_BOOT} tirages). Seuils G1 : proposition à pré-enregistrer."]
(OUT / "E0_RESULTS.md").write_text("\n\n".join(parts) + "\n", encoding="utf-8")
print("Écrit :", (OUT / "E0_RESULTS.md").resolve())

Écrit : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\outputs\e0_mortality\E0_RESULTS.md


## Et ensuite

1. Lancez d'abord `SEEDS = [0]` pour vérifier la durée et la convergence, puis les 10 graines.
2. Envoyez-moi `outputs/e0_mortality/E0_RESULTS.md`. Si G1 n'est pas atteint, on en discute avant d'aller plus loin (spécification, critères go/no-go).
3. Optionnel : `TASK = "aki"` pour la tâche secondaire (mêmes étapes, sorties séparées).
4. Notebook suivant, `04_experts_cache` : modèles des régimes R2 et R3, experts de transfert (mélange en espace logit, recalibration de Platt, harmonisation) et cache des prédictions de tous les experts.